<a href="https://colab.research.google.com/github/aladdin4220243/aass1122/blob/main/Create_Gmail.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile names.txt
ErmaKristopher
ParisRalph
KimberleeEric
KenyaShaun
AlfonsoDrew
LeahSamuel
BryannaEd
CynthiaEarnest
LeoAlvarez
CarmenEarnest

Overwriting names.txt


In [ ]:
%%writefile names.txt
DuaneMason
LucilleAlvarez
SandraBrown
NancyShirley
StanfordHill
JohnJones
JessieBenson
DonnaLamb
CorinneSkaggs
OnaAdams

Overwriting names.txt


In [ ]:
%%writefile names.txt
KarinaNeal
BarrettDuncan
MelHodges
MaryKirk
WilfredoRamsey
ChristinaMcclain
NickJacobson
LoriKnapp
SidneyYoder
PearlieDouglas

Overwriting names.txt


In [ ]:
%%writefile names.txt
ZaydenCole
KylerBrooks
BrennanTate
JaxonWells
CasonReed
LandenVaughn
TalonHayes
KianMercer
MalikWeston
JamisonKnox

Overwriting names.txt


In [ ]:
%%writefile names.txt
JamesSmith
EmmaJohnson
MichaelBrown
OliviaWilliams
DavidJones
SophiaGarcia
DanielMiller
IsabellaDavis
MatthewWilson
AvaAnderson

Overwriting names.txt


In [ ]:
%%writefile names.txt
ShelyGras
Calkeflein
IkeGardner
ChasFord
ArronBryan
ScottBallard
JulesTerry
EzekielRay
VitoClay
YoungBoyd

Overwriting names.txt


In [ ]:
%%writefile names.txt
ArvenLunox
ZarikVelto
MynorKestel
RovikTandor
ElvarNexil
DrenikSoltar
KavroMeldin
XeronTalvik
JorvenQuilax
FenricDolmar

Overwriting names.txt


In [47]:
%%writefile names.txt
EthanMBrooks
LiamJCarter
NoahDEvans
MasonKFoster
LoganTGray
LucasRHayes
OliverPJames
ElijahLKing
AidenSLee
JacksonVMartin

Overwriting names.txt


In [48]:
%%writefile names.txt
KoaAshby
FiaCoyle
PennGreer
LarkHale
BlaiseKaye
LivMoss
BreckNye
MarenReeve
KianVoss
NolaYork

Overwriting names.txt


In [44]:
%%writefile names.txt
MasonLane
MasonLee
LashondaArnold
FranciscoLeslie
PaigeHarvey
EthanDewey
SamanthaGreen
ChastityCorey
StaceyBlake
LesterMartinez

Overwriting names.txt


In [50]:
%%writefile names.txt
JamesCarter
MichaelTurner
WilliamParker
DanielBrooks
MatthewReed
JosephCollins
DavidFoster
AndrewBennett
ChristopherHayes
JoshuaMorgan

Overwriting names.txt


In [52]:
%%writefile names.txt
NickyConrad
LazaroSweeney
AlbertaCooke
AudraArroyo
MohammadEllison
EstherGillespie
MortonMiddleton
SherwoodFloyd
HoustonChase
LindaPace

Overwriting names.txt


In [53]:

import itertools
import sqlite3
import sys
import unicodedata
from pathlib import Path

# ---------------- Config ----------------
INPUT_FILE     = Path("names.txt")
OUTPUT_FOLDER  = Path("/content/orgenal folder/disabled")
DOMAIN         = "gmail.com"

MAX_LINES_PER_FILE = 500_000
MIN_USER_LEN = 6      # Gmail min username length
MAX_USER_LEN = 30     # Gmail max username length

SUFFIX_DEPTH        = 3       # append 1..3 random letters (26 + 676 + 17,576 per base)
SUFFIX_LETTERS      = "abcdefghijklmnopqrstuvwxyz"
KEEP_TRUNCATIONS    = True    # original name, remove last 1-3 chars, duplicate last char
FULLNAME_TRANSFORMS = True    # "first last" -> first.last, flast, lastfirst, ...
DEDUP               = True    # set if small, SQLite if large (memory-safe)

# Precompute suffixes once instead of rebuilding product() per name
suffixes = [""]
for depth in range(1, SUFFIX_DEPTH + 1):
    suffixes += ["".join(p) for p in itertools.product(SUFFIX_LETTERS, repeat=depth)]

def normalize(raw: str) -> str:
    raw = unicodedata.normalize("NFKD", raw)
    raw = raw.encode("ascii", "ignore").decode().lower()
    for ch in "_-'’":
        raw = raw.replace(ch, "")
    return raw

def base_names(raw: str) -> list:
    raw = normalize(raw)
    parts = raw.split()
    if len(parts) >= 2:
        first, last = parts[0], parts[-1]
        if FULLNAME_TRANSFORMS:
            bases = [f"{first}.{last}", f"{first}{last}", f"{last}{first}",
                     f"{first[0]}{last}", f"{first}.{last[0]}",
                     f"{first[0]}.{last}", f"{last}.{first}",
                     f"{first[0]}{last[0]}", first, last]
        else:
            bases = [first, last, f"{first}{last}"]
    else:
        bases = [raw] if raw else []
    return [b for b in bases if b and all(c.isalnum() or c == "." for c in b)]

def is_valid_user(u: str) -> bool:
    return (MIN_USER_LEN <= len(u) <= MAX_USER_LEN
            and u[0] != "." and u[-1] != "."
            and ".." not in u
            and all(c.isalnum() or c == "." for c in u))

def variants_for(raw: str) -> set:
    out = set()
    for base in base_names(raw):
        cands = []
        if KEEP_TRUNCATIONS:
            cands.append(base)
            for i in range(1, 4):
                if len(base) > i:
                    cands.append(base[:-i])
            cands.append(base + base[-1])
        cands += [base + s for s in suffixes]
        out.update(c for c in cands if is_valid_user(c))
    return out

def gen_direct():
    with INPUT_FILE.open(encoding="utf-8") as f:
        for line in f:
            raw = line.strip()
            if not raw:
                continue
            for u in variants_for(raw):
                yield f"{u}@{DOMAIN}"

def write_emails(emails_iter):
    OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
    file_index, written = 1, 0
    out = None
    try:
        for email in emails_iter:
            if written % MAX_LINES_PER_FILE == 0:
                if out is not None:
                    out.close()
                out = (OUTPUT_FOLDER / f"emails_{file_index}.txt").open("w", encoding="utf-8")
                file_index += 1
            out.write(email + "\n")
            written += 1
    finally:
        if out is not None:
            out.close()
    return written, file_index - 1

def main():
    OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
    names = sum(1 for ln in INPUT_FILE.open(encoding="utf-8") if ln.strip())
    if not names:
        print("[!] no names found in", INPUT_FILE, file=sys.stderr)
        return
    rough = (len(suffixes) + (3 if KEEP_TRUNCATIONS else 0)) * (10 if FULLNAME_TRANSFORMS else 1)
    est = names * rough
    print(f"[*] {names:,} names -> ~{est:,} candidate emails "
          f"(~{est * (len(DOMAIN) + 14) / 1e9:.1f} GB raw)", file=sys.stderr)

    if not DEDUP:
        written, files = write_emails(gen_direct())
    elif est <= 20_000_000:          # small enough: in-memory set
        seen = set(gen_direct())
        written, files = write_emails(sorted(seen))
    else:                            # large: SQLite dedup, memory-safe
        db_path = OUTPUT_FOLDER / "dedup.db"
        db_path.unlink(missing_ok=True)
        con = sqlite3.connect(db_path)
        con.execute("PRAGMA journal_mode=OFF")
        con.execute("PRAGMA synchronous=OFF")
        con.execute("PRAGMA cache_size=-200000")
        con.execute("CREATE TABLE emails (email TEXT PRIMARY KEY) WITHOUT ROWID")
        batch, processed = [], 0
        for email in gen_direct():
            batch.append((email,))
            if len(batch) >= 50_000:
                con.executemany("INSERT OR IGNORE INTO emails VALUES (?)", batch)
                batch.clear()
            processed += 1
            if processed % 500_000 == 0:
                print(f"[*] {processed:,} candidates inserted...", file=sys.stderr)
        if batch:
            con.executemany("INSERT OR IGNORE INTO emails VALUES (?)", batch)
        con.commit()
        n = con.execute("SELECT COUNT(*) FROM emails").fetchone()[0]
        print(f"[*] {n:,} unique emails in DB", file=sys.stderr)
        try:
            written, files = write_emails(row[0] for row in con.execute(
                "SELECT email FROM emails ORDER BY email"))
        finally:
            con.close()

    print(f"[+] done: {written:,} unique emails -> {files} file(s) in {OUTPUT_FOLDER}", file=sys.stderr)

if __name__ == "__main__":
    main()
files.download(zip_path)

[*] 10 names -> ~1,828,200 candidate emails (~0.0 GB raw)
[+] done: 182,820 unique emails -> 1 file(s) in /content/orgenal folder/disabled


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [54]:

import os

OUTPUT_FOLDER = "/content/orgenal folder/disabled"
MAX_LINES_PER_FILE = 500_000

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

file_index = 1
line_count = 0
outfile = open(
    os.path.join(OUTPUT_FOLDER, f"emails_{file_index}.txt"),
    "w",
    encoding="utf-8"
)

with open("names.txt", "r", encoding="utf-8") as infile:
    for line in infile:
        name = line.strip()
        if not name:
            continue

        for number in range(0, 100000):
            if line_count >= MAX_LINES_PER_FILE:
                outfile.close()
                file_index += 1
                line_count = 0
                outfile = open(
                    os.path.join(OUTPUT_FOLDER, f"emails_{file_index}.txt"),
                    "w",
                    encoding="utf-8"
                )

            outfile.write(f"{name}{number}@gmail.com\n".lower())
            line_count += 1

outfile.close()

print(f"Done! Files saved in: {OUTPUT_FOLDER}")

import shutil
from google.colab import files
import os # Import os module if not already imported in the context of this cell

# OUTPUT_FOLDER is available from previous cells' execution context.
# For demonstration, if it were not, you might redefine it like:
# OUTPUT_FOLDER = "/content/orgenal folder/disabled"

# Create a zip archive of the OUTPUT_FOLDER
# The first argument is the base_name for the archive file. os.path.basename(OUTPUT_FOLDER) will get 'disabled'
# The second argument is the format, 'zip'
# The third argument is the root_dir, where the archive will be created. '/' puts it in the root.
# The last argument is base_dir, the directory to be archived.
zip_path = shutil.make_archive(os.path.join('/content', os.path.basename(OUTPUT_FOLDER)), 'zip', base_dir=OUTPUT_FOLDER)

print(f"The folder '{OUTPUT_FOLDER}' has been zipped to '{zip_path}'")

# Download the zip file
files.download(zip_path)

Done! Files saved in: /content/orgenal folder/disabled
The folder '/content/orgenal folder/disabled' has been zipped to '/content/disabled.zip'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>